# Breast Cancer Classification - CBIS-DDSM ROI

DenseNet-121 ensemble classifier trained on Region of Interest (ROI) cropped mammography images.

### Architecture
- **Backbone:** DenseNet-121 (ImageNet pretrained)
- **Head:** 2048 dense units, 0.35 dropout
- **Loss:** Focal loss (gamma=2.0, alpha=0.7)
- **Ensemble:** 3 models with TTA inference

### Training Strategy
| Stage | Epochs | Learning Rate | Layers |
|-------|--------|---------------|--------|
| 1 | 30 | 1e-3 | Head only |
| 2 | 50 | 3e-4 | Top 150+ |
| 3 | 100 | 1e-4 | All |

## 1. Environment Setup

In [ ]:
import os
import sys
import gc
import json
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# Environment detection
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNTIME_ENV = 'colab'
    import subprocess
    subprocess.run(['pip', 'install', 'pydicom', '-q'], capture_output=True)
except ImportError:
    RUNTIME_ENV = 'local'

import numpy as np
import pandas as pd
import cv2
import pydicom
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC

from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# GPU configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print(f"Runtime: {RUNTIME_ENV.upper()}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {len(gpus) > 0}")

## 2. Configuration

In [ ]:
class Config:
    """Experiment configuration for ROI-based classification."""
    
    def __init__(self, gpu_memory_gb=16):
        # Paths
        if RUNTIME_ENV == 'colab':
            self.base_path = Path('/content/drive/MyDrive/CBIS-DDSM-data')
        else:
            self.base_path = Path('/home/tars/Desktop/final_project/CBIS-DDSM model training')
        
        self.dicom_path = self.base_path / 'CBIS-DDSM'
        self.csv_path = self.base_path / 'csv'
        self.cache_path = self.base_path / 'preprocessed_cache_roi'
        self.checkpoint_path = self.base_path / 'checkpoints_roi'
        self.results_path = self.base_path / 'results_roi'
        
        # Image
        self.image_size = (224, 224)
        self.clahe_clip = 2.0
        self.clahe_tile = (8, 8)
        
        # Model
        self.dense_units = 2048
        self.dropout_rate = 0.35
        self.l2_reg = 5e-4
        
        # Training
        self.batch_size = 32 if gpu_memory_gb >= 16 else 8
        self.batch_size_finetune = 16 if gpu_memory_gb >= 16 else 4
        self.ensemble_size = 3
        self.ensemble_seeds = [42, 123, 456]
        
        self.stage1_epochs = 30
        self.stage1_lr = 1e-3
        self.stage2_epochs = 50
        self.stage2_lr = 3e-4
        self.stage2_frozen = 150
        self.stage3_epochs = 100
        self.stage3_lr = 1e-4
        
        # Regularization
        self.label_smoothing = 0.1
        self.gradient_clip = 1.0
        self.focal_gamma = 2.0
        self.focal_alpha = 0.7
        self.malignant_weight = 2.5
        
        # Callbacks
        self.early_stop_patience = 30
        self.lr_reduce_patience = 12
        self.lr_reduce_factor = 0.5
        
        # Data
        self.val_split = 0.15
        self.shuffle_buffer = 1000
        
        # Mixed precision
        self.use_mixed_precision = gpu_memory_gb >= 16
        
        self.class_weights = None
        self._create_dirs()
    
    def _create_dirs(self):
        for path in [self.cache_path, self.checkpoint_path, self.results_path]:
            path.mkdir(parents=True, exist_ok=True)


config = Config(gpu_memory_gb=16)

if config.use_mixed_precision:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')

print(f"Batch size: {config.batch_size}")
print(f"Ensemble: {config.ensemble_size} models")
print(f"Mixed precision: {config.use_mixed_precision}")

## 3. Preprocessing

In [ ]:
def apply_clahe(image, clip_limit=2.0, tile_size=(8, 8)):
    """Apply Contrast Limited Adaptive Histogram Equalization."""
    if image.dtype != np.uint8:
        image = ((image - image.min()) / (image.max() - image.min() + 1e-8) * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
    return clahe.apply(image).astype(np.float32) / 255.0


def find_dicom(case_dir, base_path):
    """Locate DICOM file in nested directory structure."""
    case_path = base_path / case_dir
    if not case_path.exists():
        return None
    try:
        for d1 in case_path.iterdir():
            if d1.is_dir():
                for d2 in d1.iterdir():
                    if d2.is_dir():
                        for f in d2.iterdir():
                            if f.suffix == '.dcm':
                                return f
    except Exception:
        pass
    return None


def load_dicom(path, size=(224, 224), use_clahe=True):
    """Load and preprocess DICOM image."""
    try:
        dcm = pydicom.dcmread(str(path))
        img = dcm.pixel_array.astype(np.float32)
        
        if hasattr(dcm, 'PhotometricInterpretation'):
            if dcm.PhotometricInterpretation == 'MONOCHROME1':
                img = img.max() - img
        
        img = ((img - img.min()) / (img.max() - img.min() + 1e-8) * 255).astype(np.uint8)
        
        if use_clahe:
            img = apply_clahe(img)
        else:
            img = img.astype(np.float32) / 255.0
        
        img = cv2.resize(img, size, interpolation=cv2.INTER_LANCZOS4)
        return np.stack([img] * 3, axis=-1)
    except Exception:
        return None


def preprocess_dataset(config, csv_files):
    """Process ROI images from CSV file list."""
    images, labels = [], []
    processed = set()
    
    for csv_file in csv_files:
        if not csv_file.exists():
            continue
        
        df = pd.read_csv(csv_file)
        path_col = next((c for c in ['cropped image file path', 'ROI mask file path', 'image file path'] 
                        if c in df.columns), None)
        if not path_col:
            continue
        
        for _, row in tqdm(df.iterrows(), total=len(df), desc=csv_file.stem):
            pathology = str(row.get('pathology', '')).upper()
            label = 1 if 'MALIGNANT' in pathology else (0 if 'BENIGN' in pathology else None)
            if label is None:
                continue
            
            rel_path = row.get(path_col)
            if pd.isna(rel_path):
                continue
            
            case_dir = Path(rel_path).parts[0]
            key = (case_dir, label)
            if key in processed:
                continue
            processed.add(key)
            
            dicom_file = find_dicom(case_dir, config.dicom_path)
            if dicom_file is None:
                continue
            
            img = load_dicom(dicom_file, config.image_size)
            if img is not None:
                images.append(img)
                labels.append(label)
    
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.float32)


def create_cache(config):
    """Create preprocessed cache files."""
    train_csvs = [config.csv_path / 'calc_case_description_train_set.csv',
                  config.csv_path / 'mass_case_description_train_set.csv']
    test_csvs = [config.csv_path / 'calc_case_description_test_set.csv',
                 config.csv_path / 'mass_case_description_test_set.csv']
    
    train_images, train_labels = preprocess_dataset(config, train_csvs)
    test_images, test_labels = preprocess_dataset(config, test_csvs)
    
    if len(train_images) > 0:
        train_imgs, val_imgs, train_lbls, val_lbls = train_test_split(
            train_images, train_labels,
            test_size=config.val_split / (1 - config.val_split),
            random_state=SEED, stratify=train_labels
        )
        
        np.savez(config.cache_path / 'train_cache.npz', images=train_imgs, labels=train_lbls)
        np.savez(config.cache_path / 'val_cache.npz', images=val_imgs, labels=val_lbls)
        np.savez(config.cache_path / 'test_cache.npz', images=test_images, labels=test_labels)

## 4. Data Loading

In [ ]:
def load_data(config):
    """Load preprocessed data from cache."""
    cache_files = {
        'train': config.cache_path / 'train_cache.npz',
        'val': config.cache_path / 'val_cache.npz',
        'test': config.cache_path / 'test_cache.npz'
    }
    
    if not all(f.exists() for f in cache_files.values()):
        create_cache(config)
    
    train = np.load(cache_files['train'])
    val = np.load(cache_files['val'])
    test = np.load(cache_files['test'])
    
    return (train['images'], train['labels'],
            val['images'], val['labels'],
            test['images'], test['labels'])


train_images, train_labels, val_images, val_labels, test_images, test_labels = load_data(config)

print(f"Train: {len(train_labels)} ({train_labels.mean():.1%} malignant)")
print(f"Val:   {len(val_labels)} ({val_labels.mean():.1%} malignant)")
print(f"Test:  {len(test_labels)} ({test_labels.mean():.1%} malignant)")

## 5. Dataset Pipeline

In [ ]:
@tf.function
def augment(images):
    """GPU-accelerated augmentation."""
    images = tf.image.random_flip_left_right(images)
    images = tf.image.random_flip_up_down(images)
    images = tf.image.random_brightness(images, 0.1)
    images = tf.image.random_contrast(images, 0.9, 1.1)
    return images


def create_dataset(images, labels, config, training=True, reduced_batch=False):
    """Create tf.data.Dataset with optional augmentation."""
    def generator():
        indices = np.arange(len(images))
        if training:
            np.random.shuffle(indices)
        for i in indices:
            yield images[i], labels[i]
    
    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32)
        )
    )
    
    if training:
        ds = ds.shuffle(config.shuffle_buffer)
    
    batch_size = config.batch_size_finetune if reduced_batch else config.batch_size
    ds = ds.batch(batch_size)
    
    if training:
        ds = ds.map(lambda x, y: (augment(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    
    return ds.prefetch(tf.data.AUTOTUNE)


train_ds = create_dataset(train_images, train_labels, config, training=True)
val_ds = create_dataset(val_images, val_labels, config, training=False)

print(f"Steps per epoch: {len(train_images) // config.batch_size}")

## 6. Class Weights

In [ ]:
def compute_weights(labels, malignant_multiplier=2.5):
    """Compute balanced class weights."""
    weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=labels)
    return {0: weights[0], 1: weights[1] * malignant_multiplier}


config.class_weights = compute_weights(train_labels, config.malignant_weight)

print(f"Benign: {config.class_weights[0]:.3f}")
print(f"Malignant: {config.class_weights[1]:.3f}")

## 7. Model Architecture

In [ ]:
def focal_loss(gamma=2.0, alpha=0.7):
    """Focal loss for class imbalance."""
    @tf.function
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_weight = tf.pow(1 - p_t, gamma)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        return tf.reduce_mean(alpha_t * focal_weight * ce)
    return loss_fn


def build_model(config, seed=42):
    """Build DenseNet-121 classifier."""
    tf.random.set_seed(seed)
    
    inputs = layers.Input(shape=(224, 224, 3))
    
    base = DenseNet121(include_top=False, weights='imagenet', input_tensor=inputs)
    base.trainable = False
    
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(config.dense_units, kernel_regularizer=regularizers.l2(config.l2_reg),
                     kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(config.dropout_rate)(x)
    outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)
    
    model = models.Model(inputs, outputs)
    
    model.compile(
        optimizer=Adam(learning_rate=config.stage1_lr, clipnorm=config.gradient_clip),
        loss=focal_loss(config.focal_gamma, config.focal_alpha),
        metrics=[BinaryAccuracy(name='accuracy'), AUC(name='auc'),
                 Precision(name='precision'), Recall(name='recall')]
    )
    
    return model


def unfreeze_layers(model, num_frozen):
    """Unfreeze layers for fine-tuning."""
    for layer in model.layers[:num_frozen]:
        layer.trainable = False
    for layer in model.layers[num_frozen:]:
        layer.trainable = True


# Verify architecture
test_model = build_model(config)
trainable = sum(np.prod(v.shape) for v in test_model.trainable_weights)
print(f"Trainable parameters: {trainable:,}")
del test_model
gc.collect()

## 8. Callbacks

In [ ]:
class WarmupScheduler(Callback):
    """Learning rate warmup."""
    
    def __init__(self, warmup_epochs, start_lr, target_lr):
        super().__init__()
        self.warmup_epochs = warmup_epochs
        self.start_lr = start_lr
        self.target_lr = target_lr
    
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.start_lr + (self.target_lr - self.start_lr) * (epoch / self.warmup_epochs)
            self.model.optimizer.learning_rate.assign(lr)


def get_callbacks(config, stage, model_idx):
    """Create callbacks for training stage."""
    return [
        ModelCheckpoint(
            str(config.checkpoint_path / f'model_{model_idx}_stage{stage}_best.h5'),
            monitor='val_auc', mode='max', save_best_only=True, verbose=0
        ),
        EarlyStopping(
            monitor='val_auc', mode='max',
            patience=config.early_stop_patience, restore_best_weights=True, verbose=0
        ),
        ReduceLROnPlateau(
            monitor='val_loss', factor=config.lr_reduce_factor,
            patience=config.lr_reduce_patience, min_lr=1e-7, verbose=0
        )
    ]

## 9. Training

In [ ]:
def train_model(config, model_idx, train_ds, val_ds, train_steps, val_steps):
    """Train single model with 3-stage progressive unfreezing."""
    seed = config.ensemble_seeds[model_idx]
    model = build_model(config, seed)
    
    history = {'loss': [], 'val_loss': [], 'auc': [], 'val_auc': []}
    
    # Stage 1: Head only
    print(f"Model {model_idx + 1} - Stage 1")
    h1 = model.fit(
        train_ds, epochs=config.stage1_epochs,
        steps_per_epoch=train_steps, validation_data=val_ds, validation_steps=val_steps,
        class_weight=config.class_weights, callbacks=get_callbacks(config, 1, model_idx), verbose=1
    )
    for k in history:
        if k in h1.history:
            history[k].extend(h1.history[k])
    
    # Stage 2: Top layers
    print(f"Model {model_idx + 1} - Stage 2")
    unfreeze_layers(model, config.stage2_frozen)
    model.compile(
        optimizer=Adam(learning_rate=config.stage2_lr, clipnorm=config.gradient_clip),
        loss=focal_loss(config.focal_gamma, config.focal_alpha),
        metrics=[BinaryAccuracy(name='accuracy'), AUC(name='auc'),
                 Precision(name='precision'), Recall(name='recall')]
    )
    h2 = model.fit(
        train_ds, epochs=config.stage2_epochs,
        steps_per_epoch=train_steps, validation_data=val_ds, validation_steps=val_steps,
        class_weight=config.class_weights, callbacks=get_callbacks(config, 2, model_idx), verbose=1
    )
    for k in history:
        if k in h2.history:
            history[k].extend(h2.history[k])
    
    # Stage 3: Full fine-tune
    print(f"Model {model_idx + 1} - Stage 3")
    unfreeze_layers(model, 0)
    model.compile(
        optimizer=Adam(learning_rate=config.stage3_lr, clipnorm=config.gradient_clip),
        loss=focal_loss(config.focal_gamma, config.focal_alpha),
        metrics=[BinaryAccuracy(name='accuracy'), AUC(name='auc'),
                 Precision(name='precision'), Recall(name='recall')]
    )
    h3 = model.fit(
        train_ds, epochs=config.stage3_epochs,
        steps_per_epoch=train_steps, validation_data=val_ds, validation_steps=val_steps,
        class_weight=config.class_weights, callbacks=get_callbacks(config, 3, model_idx), verbose=1
    )
    for k in history:
        if k in h3.history:
            history[k].extend(h3.history[k])
    
    # Load best weights
    best_path = config.checkpoint_path / f'model_{model_idx}_stage3_best.h5'
    if best_path.exists():
        model.load_weights(str(best_path))
    
    return model, history


def train_ensemble(config, train_ds, val_ds, train_steps, val_steps):
    """Train ensemble of models."""
    models_list, histories = [], []
    
    for i in range(config.ensemble_size):
        model, history = train_model(config, i, train_ds, val_ds, train_steps, val_steps)
        models_list.append(model)
        histories.append(history)
        gc.collect()
        tf.keras.backend.clear_session()
        
        if i < config.ensemble_size - 1:
            model = build_model(config, config.ensemble_seeds[i])
            model.load_weights(str(config.checkpoint_path / f'model_{i}_stage3_best.h5'))
            models_list[i] = model
    
    return models_list, histories

## 10. Execute Training

In [ ]:
train_steps = len(train_images) // config.batch_size
val_steps = len(val_images) // config.batch_size

train_ds = create_dataset(train_images, train_labels, config, training=True)
val_ds = create_dataset(val_images, val_labels, config, training=False)

print(f"Training {config.ensemble_size} models")
print(f"Total epochs: {(config.stage1_epochs + config.stage2_epochs + config.stage3_epochs) * config.ensemble_size}")

ensemble_models, ensemble_histories = train_ensemble(config, train_ds, val_ds, train_steps, val_steps)

## 11. Test-Time Augmentation

In [ ]:
def tta_augment(image, idx):
    """Apply TTA augmentation by index."""
    augmentations = [
        lambda x: x,
        lambda x: tf.image.flip_left_right(x),
        lambda x: tf.image.flip_up_down(x),
        lambda x: tf.image.rot90(x, k=1),
        lambda x: tf.image.rot90(x, k=2),
        lambda x: tf.image.rot90(x, k=3),
        lambda x: tf.image.flip_left_right(tf.image.rot90(x, k=1)),
        lambda x: tf.image.flip_up_down(tf.image.rot90(x, k=1)),
    ]
    return augmentations[idx](image)


def predict_with_tta(models, images, num_aug=8, batch_size=16):
    """Ensemble predictions with TTA."""
    all_preds = []
    
    for aug_idx in range(num_aug):
        aug_images = np.array([tta_augment(img, aug_idx).numpy() for img in tf.constant(images)])
        model_preds = [m.predict(aug_images, batch_size=batch_size, verbose=0).flatten() for m in models]
        all_preds.append(np.mean(model_preds, axis=0))
    
    return np.mean(all_preds, axis=0)

## 12. Evaluation

In [ ]:
def evaluate(models, images, labels, config):
    """Comprehensive evaluation with TTA."""
    preds = predict_with_tta(models, images, batch_size=config.batch_size)
    pred_classes = (preds > 0.5).astype(int)
    
    # Metrics at 0.5 threshold
    auc = roc_auc_score(labels, preds)
    acc = accuracy_score(labels, pred_classes)
    prec = precision_score(labels, pred_classes, zero_division=0)
    rec = recall_score(labels, pred_classes, zero_division=0)
    f1 = f1_score(labels, pred_classes, zero_division=0)
    
    tn, fp, fn, tp = confusion_matrix(labels, pred_classes).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # Optimal threshold
    fpr, tpr, thresholds = roc_curve(labels, preds)
    opt_idx = np.argmax(tpr - fpr)
    opt_thresh = thresholds[opt_idx]
    
    opt_preds = (preds > opt_thresh).astype(int)
    opt_acc = accuracy_score(labels, opt_preds)
    opt_rec = recall_score(labels, opt_preds, zero_division=0)
    opt_prec = precision_score(labels, opt_preds, zero_division=0)
    opt_f1 = f1_score(labels, opt_preds, zero_division=0)
    tn_o, fp_o, fn_o, tp_o = confusion_matrix(labels, opt_preds).ravel()
    opt_spec = tn_o / (tn_o + fp_o) if (tn_o + fp_o) > 0 else 0
    
    return {
        'auc': float(auc), 'accuracy': float(acc), 'precision': float(prec),
        'sensitivity': float(rec), 'specificity': float(spec), 'f1': float(f1),
        'optimal_threshold': float(opt_thresh), 'optimal_accuracy': float(opt_acc),
        'optimal_sensitivity': float(opt_rec), 'optimal_specificity': float(opt_spec),
        'optimal_precision': float(opt_prec), 'optimal_f1': float(opt_f1),
        'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
        'predictions': preds.tolist(), 'fpr': fpr.tolist(), 'tpr': tpr.tolist()
    }

## 13. Run Evaluation

In [ ]:
# Load models from checkpoints
loaded_models = []
for i in range(config.ensemble_size):
    model = build_model(config, config.ensemble_seeds[i])
    ckpt = config.checkpoint_path / f'model_{i}_stage3_best.h5'
    if ckpt.exists():
        model.load_weights(str(ckpt))
        loaded_models.append(model)

if loaded_models:
    ensemble_models = loaded_models

results = evaluate(ensemble_models, test_images, test_labels, config)

print("Test Results (threshold=0.5)")
print("-" * 40)
print(f"AUC:         {results['auc']:.4f}")
print(f"Accuracy:    {results['accuracy']:.4f}")
print(f"Sensitivity: {results['sensitivity']:.4f}")
print(f"Specificity: {results['specificity']:.4f}")
print(f"F1:          {results['f1']:.4f}")
print()
print(f"Optimal Threshold: {results['optimal_threshold']:.3f}")
print(f"Opt Accuracy:    {results['optimal_accuracy']:.4f}")
print(f"Opt Sensitivity: {results['optimal_sensitivity']:.4f}")
print(f"Opt Specificity: {results['optimal_specificity']:.4f}")

## 14. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ROC Curve
axes[0].plot(results['fpr'], results['tpr'], 'b-', lw=2, label=f"AUC = {results['auc']:.4f}")
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Confusion Matrix
cm = results['confusion_matrix']
cm_array = np.array([[cm['tn'], cm['fp']], [cm['fn'], cm['tp']]])
sns.heatmap(cm_array, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix')

# Training History
for i, h in enumerate(ensemble_histories):
    axes[2].plot(h['val_auc'], label=f'Model {i+1}', alpha=0.7)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Validation AUC')
axes[2].set_title('Training Progress')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(config.results_path / 'evaluation.png', dpi=150)
plt.show()

## 15. Save Results

In [ ]:
# Save models
for i, model in enumerate(ensemble_models):
    model.save(str(config.results_path / f'model_{i}.h5'))

# Save results
output = {
    'experiment': 'V12_ROI_Production',
    'config': {
        'image_size': list(config.image_size),
        'batch_size': config.batch_size,
        'ensemble_size': config.ensemble_size,
        'dense_units': config.dense_units,
        'dropout_rate': config.dropout_rate,
        'focal_gamma': config.focal_gamma,
        'focal_alpha': config.focal_alpha,
    },
    'results': {
        'auc': results['auc'],
        'accuracy': results['accuracy'],
        'sensitivity': results['sensitivity'],
        'specificity': results['specificity'],
        'f1': results['f1'],
        'optimal_threshold': results['optimal_threshold'],
    },
    'confusion_matrix': results['confusion_matrix']
}

with open(config.results_path / 'results.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f"Results saved to {config.results_path}")

## 16. Summary

In [ ]:
print("=" * 50)
print("EXPERIMENT SUMMARY")
print("=" * 50)
print()
print("Configuration")
print("-" * 50)
print(f"Architecture:    DenseNet-121 + {config.dense_units} dense")
print(f"Ensemble:        {config.ensemble_size} models")
print(f"Training:        {config.stage1_epochs + config.stage2_epochs + config.stage3_epochs} epochs/model")
print()
print("Results")
print("-" * 50)
print(f"AUC:             {results['auc']:.4f}")
print(f"Accuracy:        {results['accuracy']:.4f}")
print(f"Sensitivity:     {results['sensitivity']:.4f}")
print(f"Specificity:     {results['specificity']:.4f}")
print(f"F1 Score:        {results['f1']:.4f}")
print("=" * 50)